   
Databricks notebook source
SENSE 프로젝트 — Silver Layer: KOSPI options raw → silver
담당: 1조 (정형 데이터)

목적:
  ADLS Gen2 raw/public_data/finance/options/ 에서 KOSPI 옵션 데이터를 읽어
  콜 거래량, 내재변동성(IV), 미결제약정(OI) 지표 추출 후
  curated/kfinance/ 에 저장

[데이터 특성]
  - 출처: 금융위원회 KRX 증권시장 통계 API
  - 주기: 월별 스냅샷 (28개월)
  - 파티션: year_month=YYYYMM

> ⚠️ 코스피200 풋 옵션 미포함 → PCR 계산 불가
> → 콜 옵션 기반 시장 센티먼트 지표로 대체

[추출 지표]
  call_volume   : 코스피200 콜 옵션 일별 거래량 합계
  call_oi       : 코스피200 콜 옵션 미결제약정 합계
  avg_iv        : 거래량 가중 평균 내재변동성 (VIX 대용)
  iv_change     : 내재변동성 전월비 변화량
  vol_change_pct: 거래량 전월비 변화율 (%)

# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# ============================================================
# 스토리지 계정 및 경로 상수 정의
# ============================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER    = "raw"
SILVER_CONTAINER = "curated"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CALENDAR_PATH  = f"{BASE_PATH_SILVER}/master_calendar.parquet"
KFINANCE_BRONZE_PATH = f"{BASE_PATH_BRONZE}/public_data/finance/options"
KFINANCE_SILVER_PATH    = f"{BASE_PATH_SILVER}/kfinance"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)
print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")


# 1. master_calendar 로드

옵션 데이터는 한국 영업일 기준으로 발표됩니다.
캘린더 inner join 으로 공휴일 행을 제거합니다.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date
from pyspark.sql import Window

calendar_df = (
    spark.read.parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "한국_휴장일_여부")
)

kr_biz_days = (
    calendar_df
    .filter(col("한국_휴장일_여부") == False)
    .select(col("기준일자").alias("date"))
)

print(f"✅ master_calendar 로드 완료: 한국 영업일 {kr_biz_days.count()}일")


   
# 2. KOSPI options raw 데이터 로드 및 컬럼 확인

파티션 구조: `year_month=YYYYMM/`

> 출처: 금융위원회 KRX 증권시장 통계 API

실제 컬럼 매핑:
| raw 컬럼 | 의미 | 사용 방식 |
|---|---|---|
| `basDt` | 기준일자 (yyyyMMdd) | → `date` (DateType) |
| `itmsNm` | 종목명 | → C/P 파싱 (예: "코스피200 **C** 202512 170.0") |
| `trqu` | 거래량 | → `vol` (double) |
| `prdCtg` | 상품구분 | "파생 옵션 코스피200" 고정 |

> ⚠️ 외국인 순매수 (`FORN_NET_BUY`) 컬럼은 이 API에 **포함되지 않음**

In [0]:
opt_raw_df = (
    spark.read
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(KFINANCE_BRONZE_PATH)
)

print(f"✅ options raw 로드 완료: {opt_raw_df.count()}행")
print(f"\n컬럼 목록 및 타입:")
opt_raw_df.printSchema()
print(f"\n샘플 데이터:")
display(opt_raw_df.limit(10))

   
# 3. 스키마 정리 — 코스피200 콜 옵션 지표 추출

데이터에 코스피200 풋 옵션이 없으므로 PCR 계산 불가.
대신 콜 옵션에서 유의미한 시장 센티먼트 지표를 추출합니다:

| 지표 | 계산 | 의미 |
|---|---|---|
| `call_volume` | 코스피200 C 거래량 합계 | 시장 활동성 |
| `call_oi` | 코스피200 C 미결제약정 합계 | 포지션 규모 |
| `avg_iv` | 거래량 가중평균 내재변동성 | **VIX 대용** (공포/탐욕 지표) |

> `avg_iv` = Σ(IV × 거래량) / Σ(거래량) — 거래량 0인 계약 제외

In [0]:
# ── 코스피200 콜 옵션만 필터 ───────────────────────────────
opt_kospi_call = (
    opt_raw_df
    .withColumn("date",   to_date(col("basDt"), "yyyyMMdd"))
    .withColumn("vol",    col("trqu").cast("double"))
    .withColumn("iv",     col("iptVlty").cast("double"))
    .withColumn("oi",     col("opnint").cast("double"))
    .withColumn("opt_type", F.regexp_extract(col("itmsNm"), r"\s(C|P)\s", 1))
    .filter(col("date").isNotNull())
    .filter(col("opt_type") == "C")
    .filter(col("itmsNm").startswith("코스피200"))
)

# ── 일별 집계: 거래량, 미결제약정, 가중평균 IV ─────────
opt_daily_df = (
    opt_kospi_call
    .groupBy("date")
    .agg(
        F.sum("vol").alias("call_volume"),
        F.sum("oi").alias("call_oi"),
        # 거래량 가중 평균 IV (거래량 0인 계약 제외)
        F.round(
            F.sum(F.when(col("vol") > 0, col("iv") * col("vol")))
            / F.sum(F.when(col("vol") > 0, col("vol"))),
            2
        ).alias("avg_iv")
    )
    .orderBy("date")
)

print(f"✅ 코스피200 콜 지표 추출 완료: {opt_daily_df.count()}행")
display(opt_daily_df.limit(10))

# 4. 한국 영업일 필터 (master_calendar 기반)


In [0]:
opt_filtered_df = (
    opt_daily_df
    .join(kr_biz_days, on="date", how="inner")
)

before = opt_daily_df.count()
after  = opt_filtered_df.count()
print(f"✅ 한국 영업일 필터 완료")
print(f"   필터 전: {before}일 → 필터 후: {after}일 (제거: {before-after}일)")


   
# 5. 파생 컬럼 생성

| 컬럼명 | 계산 | 의미 |
|---|---|---|
| `iv_change` | avg_iv - lag(avg_iv) | 내재변동성 전월비 변화량 |
| `vol_change_pct` | (거래량 - 전월) / 전월 × 100 | 거래량 전월비 변화율 |
| `iv_surge_flag` | avg_iv가 전월 대비 +5 이상 상승 시 1 | 변동성 급등 신호 |

> 월별 데이터이므로 Forward Fill 전에 변화량 계산

In [0]:
w_monthly = Window.orderBy("date")

opt_featured_df = (
    opt_filtered_df

    # ── IV 전월비 변화량 ────────────────────────────────────
    .withColumn(
        "iv_change",
        F.round(col("avg_iv") - F.lag("avg_iv", 1).over(w_monthly), 2)
    )

    # ── 거래량 전월비 변화율 (%) ───────────────────────────
    .withColumn(
        "vol_change_pct",
        F.round(
            (col("call_volume") - F.lag("call_volume", 1).over(w_monthly))
            / F.lag("call_volume", 1).over(w_monthly) * 100,
            2
        )
    )

    # ── IV 급등 신호 (avg_iv 전월비 +5 이상 상승) ──────────
    .withColumn(
        "iv_surge_flag",
        F.when(col("iv_change") >= 5.0, F.lit(1)).otherwise(F.lit(0))
    )

    # ── 첫 행 null → 0 ───────────────────────────────────
    .na.fill(0.0, subset=["iv_change", "vol_change_pct"])

    # ── 파티션 컬럼 ──────────────────────────────────────
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
)

print("✅ 파생 컬럼 생성 완료")
display(
    ㅊ
    .select("date", "call_volume", "call_oi", "avg_iv", "iv_change", "vol_change_pct", "iv_surge_flag")
    .orderBy("date")
    .limit(15)
)

# 6. 데이터 검증


In [0]:
date_range = opt_featured_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.count("date").alias("영업일수")
).collect()[0]

print("=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  영업일수: {date_range['영업일수']}일")

total = opt_featured_df.count()
print(f"\n=== 🔍 컬럼별 null 비율 ===")
for c in ["call_volume", "call_oi", "avg_iv", "iv_change", "vol_change_pct", "iv_surge_flag"]:
    null_cnt = opt_featured_df.filter(col(c).isNull()).count()
    status = "✅" if null_cnt == 0 else "⚠️"
    print(f"  {status} {c}: null {null_cnt}건")

surge_cnt = opt_featured_df.filter(col("iv_surge_flag") == 1).count()
print(f"\n=== 🚨 IV 급등 신호: {surge_cnt}건 ===")
if surge_cnt > 0:
    display(
        opt_featured_df.filter(col("iv_surge_flag") == 1)
        .select("date", "avg_iv", "iv_change", "call_volume")
        .orderBy("date")
    )

# 7. Silver 저장


In [0]:
(
    opt_featured_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(KFINANCE_SILVER_PATH)
)

saved_files = dbutils.fs.ls(KFINANCE_SILVER_PATH)
print(f"✅ Silver 저장 완료")
print(f"   경로     : {KFINANCE_SILVER_PATH}")
print(f"   파티션 수: {len(saved_files)}개")


# 8. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(SILVER_PATH)
print(f"✅ silver 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼: {df_check.columns}")
display(df_check.orderBy("date").limit(10))


   
## ✅ 완료 후 다음 단계

### `opt_featured_df` 요약 (28행 × 9컬럼, 월별 스냅샷)
| # | 컬럼 | 타입 | nullable | 설명 | 예시 |
|---|---|---|---|---|---|
| 1 | `date` | DateType | O | 기준일자 (해당 월 마지막 영업일) | 2024-09-27 |
| 2 | `call_volume` | DoubleType | O | 코스피200 콜 거래량 합계 — 시장 활동성 | 472,405 |
| 3 | `call_oi` | DoubleType | O | 코스피200 콜 미결제약정 합계 — 포지션 규모 | 606,357 |
| 4 | `avg_iv` | DoubleType | O | 거래량 가중평균 내재변동성 — **VIX 대용** | 26.73 |
| 5 | `iv_change` | DoubleType | X | avg_iv 전월비 변화량, 첫 행 0 | +8.70 |
| 6 | `vol_change_pct` | DoubleType | X | 거래량 전월비 변화율 (%), 첫 행 0 | +36.40 |
| 7 | `iv_surge_flag` | IntegerType | X | IV 급등 신호 (전월비 +5 이상 상승 시 1) | 1 |
| 8 | `year` | IntegerType | O | 파티션 컬럼 (연도) | 2024 |
| 9 | `month` | IntegerType | O | 파티션 컬럼 (월) | 9 |

**기간**: 2024-01-30 ~ 2026-04-07 (28개월) · null 0건 · IV 급등 5건 검출

---

> **Gold JOIN 시 참고:**
> - 월별 28개 데이터포인트 → Forward Fill로 일별 확장 필요
> - `date` = 한국 영업일 기준